[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_5_qlearning_complete.ipynb)

<div style="text-align:center">
    <h1>
        Q-Learning
    </h1>
</div>

<br><br>

<div style="text-align:center">
    In this notebook we are going to implement a method that learns from experience and uses bootstrapping.
    It is known as Q-Learning.
</div>

<br>

<div style="text-align:center">
    This method follows an off-policy strategy, in which we'll use an exploratory policy $b(s)$ to interact with the environment and a target policy $\pi(s)$ that will participate in the learning process.
</div>


<br>


In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_policy, plot_action_values


## Import the necessary software libraries:

In [ ]:
# NumPy gives us fast arrays - we store the action-value table as one.
import numpy as np
# Matplotlib is the plotting library used by the helper drawing functions.
import matplotlib.pyplot as plt

## Create the environment, value table and policy

#### Create the environment

In [ ]:
env = Maze()

#### Create the $Q(s, a)$ table

In [ ]:
# The Q-table holds one value for every (state, action) pair.
# The maze is a 5x5 grid (state = row, column) with 4 actions, so shape (5, 5, 4).
# Every value starts at 0 because the agent has learned nothing yet.
action_values = np.zeros((5, 5, 4))

#### Create the target policy $\pi(s)$

In [ ]:
# The TARGET policy is the behaviour we ultimately want to learn: pure greedy.
# Given a state, it always picks the action with the highest estimated value.
def target_policy(state):
    # Look up the action-values for this state.
    av = action_values[state]
    # Pick the best action; if several tie for the top value, choose at random
    # among them so we do not always favour the same one.
    return np.random.choice(np.flatnonzero(av == av.max()))

#### Create the exploratory policy $b(s)$

In [ ]:
# The EXPLORATORY (behaviour) policy is what actually drives the agent around
# the maze while it learns. Here it is fully random - it tries every action
# equally so the whole maze gets explored.
def exploratory_policy(state):
    # Return a random action (0..3) regardless of the state.
    return np.random.randint(4)

#### Plot the value table $Q(s,a)$

In [ ]:
plot_action_values(action_values)

#### Plot the policy

In [ ]:
plot_policy(action_values, env.render(mode='rgb_array'))

## Implement the algorithm

### How Q-Learning learns: off-policy temporal-difference control

Like SARSA, Q-Learning is a **temporal-difference (TD)** method: it updates its
value estimates **after every step** instead of waiting for the episode to end,
and it **bootstraps** (it learns from its own current guess of the next state's value).

**The key new idea: off-policy.** Q-Learning uses **two** policies:

- a **behaviour policy** $b(s)$ that decides how the agent actually *moves* (here a
  fully random, highly exploratory policy), and
- a **target policy** $\pi(s)$ that it is actually *learning about* (here a pure
  greedy policy - always take the best-known action).

Because the agent explores with one policy but learns the value of a different,
greedy one, Q-Learning is called **off-policy**. This lets it explore freely while
still learning the value of behaving optimally.

**The Q-Learning update.** From state $S_t$ we take action $A_t$, get reward
$R_{t+1}$, and land in $S_{t+1}$. We then update using the **best** action available
in the next state:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha\,\bigl[\,R_{t+1} + \gamma\,\max_{a} Q(S_{t+1}, a) - Q(S_t, A_t)\,\bigr]$$

**SARSA vs Q-Learning - the one-line difference.** Both move $Q(S_t,A_t)$ toward a
TD target. The only change is the next-state term:

- **SARSA (on-policy):** uses $Q(S_{t+1}, A_{t+1})$ - the action the policy *will
  actually take* next.
- **Q-Learning (off-policy):** uses $\max_{a} Q(S_{t+1}, a)$ - the *best possible*
  next action, regardless of what is taken.

In this notebook the target policy is greedy, so picking its action is the same as
taking the $\max$ - that is how the code below implements the rule.

In [ ]:
# Q-Learning trains the Q-table by exploring with one policy (exploratory_policy)
# while learning the value of a different, greedy policy (target_policy).
# Arguments:
#   action_values     : the Q-table being trained (updated in place)
#   exploratory_policy: the behaviour policy used to move around (random here)
#   target_policy     : the greedy policy whose values we are learning
#   episodes          : number of full runs through the maze
#   alpha             : learning rate (step size of each update)
#   gamma             : discount factor (how much future reward counts)
def q_learning(action_values, exploratory_policy, target_policy,
               episodes, alpha=0.1, gamma=0.99):

    # One pass of the loop per training episode.
    for episode in range(1, episodes + 1):
        # Reset the maze to its starting state.
        state = env.reset()
        # 'done' turns True when the agent reaches the goal.
        done = False

        # Step through the maze until the episode ends.
        while not done:
            # Choose how to MOVE using the exploratory (random) policy.
            action = exploratory_policy(state)
            # Take that action; get the next state, reward, and done flag.
            next_state, reward, done, _ = env.step(action)
            # Choose the next action with the GREEDY target policy. We do NOT
            # necessarily take this action - we only use it to learn. This is
            # what makes Q-Learning "off-policy": we explore one way but learn
            # the value of acting greedily.
            next_action = target_policy(next_state)

            # Current estimate for the (state, action) we just took.
            qsa = action_values[state][action]
            # Value of the greedy next action - effectively the MAX over next
            # actions, since the target policy always picks the best one.
            next_qsa = action_values[next_state][next_action]
            # Q-Learning update: move the old estimate toward the TD target
            #   reward + gamma * next_qsa
            # Using a greedy (max) next value is the key difference from SARSA,
            # and like SARSA this is bootstrapping (learning from an estimate).
            action_values[state][action] = qsa + alpha * (reward + gamma * next_qsa - qsa)

            # Advance to the next state for the following loop iteration.
            state = next_state

In [ ]:
# Train the agent for 1000 episodes, filling in the Q-table.
q_learning(action_values, exploratory_policy, target_policy, 1000)

## Show results

#### Show resulting value table $Q(s,a)$

In [ ]:
plot_action_values(action_values)

#### Show resulting policy $\pi(\cdot|s)$

In [ ]:
plot_policy(action_values, env.render(mode='rgb_array'))

#### Test the resulting agent

In [ ]:
test_agent(env, target_policy)

## Resources

[[1] Reinforcement Learning: An Introduction. Ch. 6: Temporal difference learning](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)